In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    from thesis_utils.paths import (
        DATA_ROOT,
        RAW_DATA_DIR,
        PROCESSED_DATA_DIR,
        EVALUATION_DATA_DIR,
        ensure_data_directories,
    )
except Exception as e:
    raise RuntimeError(f"Yol import edilemedi: {e}") from e

ensure_data_directories()

print("Root:", ROOT)
print("Data root:", DATA_ROOT)


In [ ]:
import pandas as pd
from IPython.display import display, Markdown

candidate_datasets = [
    {
        "name": "FiQA 2018",
        "category": "Academic benchmark",
        "source_type": "Financial QA / aspect sentiment",
        "reason": "Finans NLP literatüründe güçlü benchmark; Reuters dışı; tezde external benchmark olarak güçlü.",
        "translation_fit": 9,
        "academic_acceptance": 10,
        "novelty_vs_current": 10,
        "external_test_fit": 10,
        "score": 9.8,
    },
    {
        "name": "Financial News Headlines (Kaggle)",
        "category": "Headline dataset",
        "source_type": "News headlines",
        "reason": "Haber başlığı formatı; Reuters dışı; Türkçeye çevrilmesi kolay; etiketsiz sürümde kendi anotasyon yapılabilir.",
        "translation_fit": 9,
        "academic_acceptance": 6,
        "novelty_vs_current": 9,
        "external_test_fit": 8,
        "score": 8.8,
    },
    {
        "name": "NewsMTSC Financial",
        "category": "Target/aspect sentiment",
        "source_type": "Financial news",
        "reason": "Yüksek kaliteli uzman annotasyonu; target-based model testleri için güçlü aday.",
        "translation_fit": 8,
        "academic_acceptance": 8,
        "novelty_vs_current": 8,
        "external_test_fit": 8,
        "score": 8.2,
    },
    {
        "name": "Financial News Articles (CNBC/Yahoo/Bloomberg)",
        "category": "Real-world news",
        "source_type": "Articles",
        "reason": "Gerçek dünya verisi; etiketsiz olduğundan kendi etiketleme gerektirir.",
        "translation_fit": 8,
        "academic_acceptance": 6,
        "novelty_vs_current": 8,
        "external_test_fit": 8,
        "score": 7.8,
    },
    {
        "name": "Reuters TRC2",
        "category": "Reuters-based",
        "source_type": "News articles",
        "reason": "Reuters zaten mevcut; yeni katkısı sınırlı; tezde zayıf ayrışım sunabilir.",
        "translation_fit": 7,
        "academic_acceptance": 8,
        "novelty_vs_current": 4,
        "external_test_fit": 6,
        "score": 6.6,
    },
    {
        "name": "All The News 2",
        "category": "Large news corpus",
        "source_type": "News corpus",
        "reason": "Büyük fakat gürültülü; finans alanı filtreleme ve temizleme gerektirir.",
        "translation_fit": 7,
        "academic_acceptance": 5,
        "novelty_vs_current": 7,
        "external_test_fit": 6,
        "score": 6.4,
    },
]

comparison_df = pd.DataFrame(candidate_datasets).sort_values("score", ascending=False).reset_index(drop=True)

# Görsel ve sunum amaçlı çıktı

display(Markdown("## Veri seti karşılaştırma tablosu"))
display(comparison_df[["name", "score", "academic_acceptance", "translation_fit", "external_test_fit", "reason"]])

display(Markdown("## En güçlü adaylar"))
for _, row in comparison_df.head(3).iterrows():
    display(Markdown(f"### {row['name']}\n- Puan: {row['score']}\n- Neden: {row['reason']}"))

# Hoca sunumu için kısa metin
summary_text = """
Önerilen sıralama:
1. FiQA 2018 — en güçlü akademik ve bağımsız dış test adayı.
2. Financial News Headlines (Kaggle) — haber başlığı formatı, Reuters dışı ve Türkçeye çevrilmesi kolay.
3. NewsMTSC Financial — uzman annotasyonu ve target-based test için güçlü ancak plain sentiment tezinde daha az doğrudan uygun.
"""

print(summary_text)


In [ ]:
import os
from pathlib import Path
import requests
import pandas as pd
from IPython.display import display, Markdown

# Dış veri setleri için indirilecek/önerilecek kaynak planı
external_candidates = [
    {
        "name": "FiQA 2018",
        "slug": "fiqa2018",
        "download_method": "manual_or_repo",
        "download_note": "FiQA 2018 resmi sayfa/repodan indirilebilir; akademik benchmark olarak en güçlü adaydır.",
        "expected_path": "fiqa2018/",
    },
    {
        "name": "Financial News Headlines (Kaggle)",
        "slug": "financial_news_headlines",
        "download_method": "kaggle",
        "download_note": "Kaggle API ile indirilebilir: kaggle datasets download -d notlucasp/financial-news-headlines",
        "expected_path": "financial_news_headlines/",
    },
    {
        "name": "NewsMTSC Financial",
        "slug": "newsmtsc_financial",
        "download_method": "manual_or_hf",
        "download_note": "Hugging Face veya resmi kaynak üzerinden alınabilir; target-based test için uygundur.",
        "expected_path": "newsmtsc_financial/",
    },
    {
        "name": "Financial News Articles (CNBC/Yahoo/Bloomberg)",
        "slug": "financial_news_articles",
        "download_method": "manual_curation",
        "download_note": "Etiketli sürüm yoksa elle toplanmalı ve anotasyon yapılmalıdır.",
        "expected_path": "financial_news_articles/",
    },
]

external_plan_df = pd.DataFrame(external_candidates)
external_plan_df = external_plan_df[["name", "download_method", "download_note", "expected_path"]]

output_dir = DATA_ROOT / "external_candidates"
output_dir.mkdir(parents=True, exist_ok=True)
external_plan_df.to_csv(output_dir / "candidate_dataset_plan.csv", index=False)

display(Markdown("## İndirme ve hazırlık planı"))
display(external_plan_df)

print(f"Plan dosyası kaydedildi: {output_dir / 'candidate_dataset_plan.csv'}")


# Yeni veri seti arama ve karşılaştırma not defteri

Bu notebook, tezde kullanılabilecek yeni dış veri setlerini sistematik biçimde değerlendirmek için hazırlanmıştır.

Amaç:
- FinBERT eğitim verisiyle çakışmaması
- Mevcut veri setlerinden farklı olması
- Akademik olarak kabul görmesi
- Türkçeye çevrilmeye uygun olması
- Tezde bağımsız dış test olarak savunulabilmesi

Sunum akışı:
1. Aday veri setlerini listele
2. Kriterlere göre puanla
3. En güçlü adayları özetle
4. Hoca sunumu için hazır metin üret
